# ML-07b — Identifiants sémantiques : recommander un item que personne n'a noté

**Navigation** : [Index](README.md) | [<< ML-07](ML-07-Recommendation-Python.ipynb) | [Suivant >> ML-08-Clustering](ML-08-Clustering-Python.ipynb)

Ce carnet prolonge [ML-07-Recommendation-Python](ML-07-Recommendation-Python.ipynb). La factorisation matricielle y prédit des notes à partir de la **matrice d'interactions** : un item que personne n’a encore noté ne fournit aucun signal d’interaction permettant d’apprendre ses coordonnées. Un score par défaut, un départage ou un autre signal peut toutefois le faire entrer dans un classement. C'est la faiblesse structurelle du filtrage collaboratif, et c'est précisément ce que la génération de candidats par le contenu cherche à couvrir.

Les recommandateurs génératifs récents franchissent un pas de plus : ils **tokenisent chaque item en un identifiant sémantique** — une suite de codes obtenue en quantifiant un plongement de son contenu — puis apprennent à générer ou à ranger ces identifiants. Le rapport technique Sona (équipe Yandex Music, [arXiv:2608.11015](https://arxiv.org/abs/2608.11015)) documente ce dispositif en production : un encodeur multimodal figé produit le plongement, un objectif contrastif collaboratif l'ajuste, une quantification résiduelle à plusieurs niveaux le compresse en tuples de codes.

Le carnet isole la question mesurable, et rien d'autre : **que gagne-t-on, et que perd-on, à ranger les items dans un espace d'identifiants quantifiés plutôt que directement dans l'espace de contenu ?** La réponse est mesurée, contre des baselines qui ne sont pas des hommes de paille.

**Crédit** : le paradigme et son vocabulaire (identifiant sémantique, quantification résiduelle à plusieurs niveaux, compression d'historique) viennent du rapport Sona. Aucune valeur du papier n'est recopiée ici : tout ce qui est chiffré dans ce carnet est calculé par ses propres cellules.

**Objectifs** : construire et quantifier un plongement de contenu ; comparer quatre classements sur des cibles chaudes/froides ; interpréter des contrastes appariés et le coût des codes.

**Prérequis** : ML-07, tableaux C#, produit scalaire et notions de SVD/KMeans. **Durée indicative** : 90 minutes.

**Navigation** : [Version Python](ML-07b-SemanticIDs-Recommandation-Python.ipynb) · [Index](README.md).


**Kernel** : .NET C# ; MathNet pour les tirages, Python.NET pour les vrais SVD/KMeans sklearn et le témoin SciPy. Corpus .NET de graine42 distinct du corpus NumPy : mêmes paramètres et budgets, valeurs différentes. SVD collaborative calculée en float32, puis U/V transférés et produit en double C# ; aucune égalité bit à bit des rangs n’est revendiquée. Définir PYTHONNET_PYDLL et les chemins CPython de son environnement.


## Ce que le carnet mesure, en une phrase

Quatre représentations d'items — populaire, collaborative, contenu, quantifiée — sont confrontées sur **deux populations de cible distinctes** : les items chauds (présents dans les interactions d'entraînement) et les items froids (présents au catalogue, absents de l'entraînement). Le résultat intéresse les deux, et le désaccord entre les deux est le contenu pédagogique.

In [1]:
#r "nuget: MathNet.Numerics, 5.0.0"
#r "nuget: PythonNet, 3.0.5"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using System.Text.Json;
using MathNet.Numerics.Distributions;
using Python.Runtime;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
if (!PythonEngine.IsInitialized) {
    Runtime.PythonDLL = Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");
    PythonEngine.Initialize();
    PythonEngine.BeginAllowThreads();
}
var rng = new Random(42);
double Dot(double[] a, double[] b) => a.Zip(b, (x,y) => x*y).Sum();
double[] Unit(double[] a) { double n=Math.Sqrt(Dot(a,a)); return a.Select(v=>v/Math.Max(n,1e-12)).ToArray(); }
double[] Dirichlet(int n,double alpha) { var a=Enumerable.Range(0,n).Select(_=>Gamma.Sample(rng,alpha,1)).ToArray(); double z=a.Sum(); return a.Select(v=>v/z).ToArray(); }
int[] SampleWeighted(double[] weights,int count) {
    var w=(double[])weights.Clone(); var result=new int[count];
    for(int j=0;j<count;j++){double total=w.Sum(),t=rng.NextDouble()*total;int k=0;for(;k<w.Length-1 && t>=w[k];k++)t-=w[k];result[j]=k;w[k]=0;}
    return result;
}
T[] Pick<T>(T[] source,int n) {
    if(n<0||n>source.Length)throw new ArgumentException("taille de tirage invalide");
    var values=(T[])source.Clone();
    for(int i=0;i<n;i++){int j=rng.Next(i,values.Length);(values[i],values[j])=(values[j],values[i]);}
    return values.Take(n).ToArray();
}
PyObject ArrayPy(double[][] values,bool single=false) {
    using(Py.GIL()) {dynamic json=Py.Import("json"),np=Py.Import("numpy"); using PyObject v=json.loads(JsonSerializer.Serialize(values)); return np.array(v,dtype:single?np.float32:np.float64);}
}
(double[][] Values,double[][] Components,double Explained) Svd(double[][] values,int dim,bool single=false) {
    using(Py.GIL()){dynamic dec=Py.Import("sklearn.decomposition"); using PyObject x=ArrayPy(values,single); using PyObject s=dec.TruncatedSVD(n_components:dim,random_state:42); using PyObject e=((dynamic)s).fit_transform(x); double[][] z=((dynamic)e).tolist().As<double[][]>();double[][] components=((dynamic)s).components_.tolist().As<double[][]>();double explained=(double)((dynamic)s).explained_variance_ratio_.sum();return(z,components,explained);}
}
using(Py.GIL()){dynamic sys=Py.Import("sys"),np=Py.Import("numpy"),sk=Py.Import("sklearn"),sp=Py.Import("scipy");Console.WriteLine($".NET {Environment.Version} | CPython {sys.version} | NumPy {np.__version__} | sklearn {sk.__version__} | SciPy {sp.__version__} | MathNet {typeof(Gamma).Assembly.GetName().Version} | Python.NET {typeof(Py).Assembly.GetName().Version}");}
Console.WriteLine("Calculs et corpus en C# ; SVD/KMeans réels sklearn par Python.NET ; RNG .NET graine42 (distinct de NumPy).");


Installed Packages MathNet.Numerics, 5.0.0 PythonNet, 3.0.5

.NET 10.0.14 | CPython 3.13.16 (main, Oct 14 2026, 01:04:01) [Clang 23.1.3 ] | NumPy 2.4.4 | sklearn 1.9.1 | SciPy 1.17.0 | MathNet 5.0.0.0 | Python.NET 3.0.5.0


Calculs et corpus en C# ; SVD/KMeans réels sklearn par Python.NET ; RNG .NET graine42 (distinct de NumPy).


### Pourquoi un corpus contrôlé

Un jeu de données réel ne dirait pas *pourquoi* un modèle gagne : les facteurs — qualité du contenu, densité des interactions, part d'items froids — y sont confondus. On génère donc un corpus dont la **vérité terrain est connue puis cachée aux algorithmes**, exactement comme [ML-08-Clustering-Python](ML-08-Clustering-Python.ipynb) génère sa vérité terrain par un mélange gaussien avant de la retirer.

La structure générative est volontairement simple et déclarée : chaque item appartient à un mélange de quelques styles latents et porte des étiquettes textuelles tirées du vocabulaire de ces styles ; chaque utilisateur a un goût, c'est-à-dire une distribution sur les mêmes styles. Les interactions sont ensuite tirées selon l'accord goût-contenu, pondéré par une popularité log-normale. **Le contenu est donc informatif par construction** — c'est la condition pour que la question posée ait un sens ; un corpus où le contenu serait du bruit ne mesurerait que le bruit.

In [2]:
var STYLES = JsonSerializer.Deserialize<Dictionary<string,string[]>>("{\"rock\": [\"guitare\", \"distorsion\", \"riff\", \"batterie\", \"power\", \"anthem\", \"grunge\", \"punk\", \"solo\", \"ampli\", \"refrain\", \"indie\"], \"electro\": [\"synthé\", \"basse\", \"bpm\", \"techno\", \"house\", \"sequencer\", \"808\", \"club\", \"loop\", \"modulaire\", \"drop\", \"acid\"], \"jazz\": [\"saxophone\", \"improvisation\", \"swing\", \"contrebasse\", \"cuivres\", \"bop\", \"standard\", \"piano\", \"trio\", \"balade\", \"blue\", \"note\"], \"classique\": [\"orchestre\", \"cordes\", \"sonate\", \"symphonie\", \"quatuor\", \"baroque\", \"chœur\", \"conducteur\", \"opus\", \"adagio\", \"fugue\", \"prélude\"], \"rap\": [\"flow\", \"beat\", \"punchline\", \"sample\", \"boom\", \"bap\", \"couplet\", \"egotrip\", \"prod\", \"street\", \"mixtape\", \"clash\"], \"pop\": [\"refrain\", \"hook\", \"radio\", \"mélodie\", \"chorus\", \"hit\", \"single\", \"voix\", \"production\", \"catchy\", \"estival\", \"dance\"], \"folk\": [\"acoustique\", \"guitare\", \"harmonica\", \"ballade\", \"conte\", \"racines\", \"campagne\", \"voix\", \"cordes\", \"americana\", \"bluegrass\", \"nylon\"], \"metal\": [\"double\", \"growl\", \"blast\", \"riff\", \"downpicking\", \"heavy\", \"thrash\", \"doom\", \"breakdown\", \"tremolo\", \"forge\", \"growler\"], \"reggae\": [\"skank\", \"dub\", \"roots\", \"basse\", \"one\", \"drop\", \"toasting\", \"rocksteady\", \"echo\", \"delay\", \"sound\", \"system\"], \"ambient\": [\"nappe\", \"drone\", \"texture\", \"réverbération\", \"minimal\", \"modulaire\", \"paysage\", \"lent\", \"espace\", \"calme\", \"field\", \"recording\"], \"world\": [\"percussions\", \"polyrythmie\", \"traditionnel\", \"kora\", \"tabla\", \"gamelan\", \"chant\", \"modes\", \"acoustique\", \"rituel\", \"fête\", \"danse\"], \"chanson\": [\"texte\", \"interprète\", \"couplet\", \"refrain\", \"piano\", \"voix\", \"auteur\", \"storytelling\", \"mélodie\", \"scène\", \"émotion\", \"français\"]}")!;
string[] styleNames=STYLES.Keys.ToArray();
string[] ERAS={"60s","70s","80s","90s","00s","10s","20s"};string[] FORMATS={"single","album","live","remix","instrumental"};
const int N_ITEMS=2000,N_USERS=800,N_INTER_PER_USER=40;
var itemMix=Enumerable.Range(0,N_ITEMS).Select(_=>Dirichlet(styleNames.Length,.25)).ToArray();
var itemPrimary=itemMix.Select(a=>Array.IndexOf(a,a.Max())).ToArray();
var itemPopularity=Enumerable.Range(0,N_ITEMS).Select(_=>LogNormal.Sample(rng,0,.6)).ToArray();
var metadata=new string[N_ITEMS];
for(int i=0;i<N_ITEMS;i++){int[] order=Enumerable.Range(0,styleNames.Length).OrderByDescending(j=>itemMix[i][j]).ToArray();var words=new List<string>();words.AddRange(Pick(STYLES[styleNames[order[0]]],4));words.AddRange(Pick(STYLES[styleNames[order[1]]],2));words.Add(ERAS[rng.Next(ERAS.Length)]);words.Add(FORMATS[rng.Next(FORMATS.Length)]);metadata[i]=string.Join(' ',words);}
var userPref=Enumerable.Range(0,N_USERS).Select(_=>Dirichlet(styleNames.Length,.35)).ToArray();
var interactions=new List<(int User,int Item)>();
for(int u=0;u<N_USERS;u++){var weights=Enumerable.Range(0,N_ITEMS).Select(i=>Dot(userPref[u],itemMix[i])*itemPopularity[i]).ToArray();foreach(int i in SampleWeighted(weights,N_INTER_PER_USER))interactions.Add((u,i));}
Console.WriteLine($"catalogue {N_ITEMS} items | utilisateurs {N_USERS} | interactions {interactions.Count} | densité {interactions.Count/(double)(N_ITEMS*N_USERS):F4}");
Console.WriteLine($"vocabulaire {STYLES.Values.SelectMany(v=>v).Concat(ERAS).Concat(FORMATS).Distinct().Count()} mots");
for(int i=0;i<3;i++)Console.WriteLine($"item {i,4} | style dominant {styleNames[itemPrimary[i]]} | {metadata[i]}");
if(interactions.Count!=32000 || interactions.GroupBy(p=>p.User).Any(g=>g.Select(p=>p.Item).Distinct().Count()!=40))throw new Exception("échantillon sans remise incorrect");


catalogue 2000 items | utilisateurs 800 | interactions 32000 | densité 0.0200


vocabulaire 141 mots


item    0 | style dominant rap | clash beat mixtape street loop basse 90s live


item    1 | style dominant folk | bluegrass harmonica voix racines flow mixtape 20s instrumental


item    2 | style dominant pop | radio dance hit single ballade bluegrass 60s live


Lecture de la sortie : le catalogue est **dense en contenu** (chaque item porte des étiquettes informatives) et **creux en interactions** (moins de trois pour cent des cases de la matrice). Cette asymétrie est la condition même du problème de démarrage à froid : le contenu voit tous les items, la matrice n'en voit qu'une partie.

L'étape suivante fabrique les deux populations de cible. Les items froids sont retirés **entièrement** de l'entraînement — aucune interaction ne les représente — mais leurs interactions réelles servent de cible de test. C'est le protocole honnête du démarrage à froid : l'item existe au catalogue et intéresse des utilisateurs, il manque seulement à la matrice.

In [3]:
int nItemsCold=(int)Math.Round(.15*N_ITEMS);
int[] coldItems=Pick(Enumerable.Range(0,N_ITEMS).ToArray(),nItemsCold);
var coldSet=coldItems.ToHashSet();
var coldTarget=new Dictionary<int,int>();var warmTarget=new Dictionary<int,int>();
var warmByUser=Enumerable.Range(0,N_USERS).Select(_=>new List<int>()).ToArray();
int coldPairsCount=0;
foreach(var pair in interactions){if(coldSet.Contains(pair.Item)){coldPairsCount++;coldTarget.TryAdd(pair.User,pair.Item);}else warmByUser[pair.User].Add(pair.Item);}
var seen=Enumerable.Range(0,N_USERS).Select(_=>new HashSet<int>()).ToArray();
for(int u=0;u<N_USERS;u++){var warm=warmByUser[u];if(warm.Count>0){warmTarget[u]=warm[^1];seen[u].UnionWith(warm.Take(warm.Count-1));}}
Console.WriteLine($"items froids {nItemsCold}/{N_ITEMS} ; interactions train {seen.Sum(s=>s.Count)} ; cibles chaudes {warmTarget.Count} ; froides {coldTarget.Count} ; interactions froides retirées {coldPairsCount}");
if(seen.Any(s=>s.Overlaps(coldSet)))throw new Exception("fuite d'items froids dans train");
Console.WriteLine("Aucune interaction froide dans l'entraînement ; mêmes candidats non consommés pour les deux populations.");


items froids 300/2000 ; interactions train 26268 ; cibles chaudes 800 ; froides 799 ; interactions froides retirées 4932


Aucune interaction froide dans l'entraînement ; mêmes candidats non consommés pour les deux populations.


Les populations chaude et froide sont partiellement communes ; leurs effectifs sont affichés séparément. Un utilisateur contribue à chaque population pour laquelle une cible est disponible. Les deux évaluations utilisent ses mêmes candidats non consommés et la même matrice d’entraînement, mais des cibles de natures différentes.

## Le plongement de contenu

Le premier étage est celui que Sona confie à un encodeur multimodal figé. Ici le contenu est du texte : on le vectorise par sac de mots sur un vocabulaire explicite, pondéré par la fréquence inverse du document, puis réduit par décomposition en valeurs singulières. Les étiquettes de style sont ce qui doit survivre à la réduction — la cellule suivante le vérifie en clair plutôt que de le supposer.

In [4]:
string[] VOCAB=STYLES.Values.SelectMany(v=>v).Concat(ERAS).Concat(FORMATS).Distinct().OrderBy(v=>v,StringComparer.Ordinal).ToArray();
var vocabIndex=VOCAB.Select((v,i)=>(v,i)).ToDictionary(t=>t.v,t=>t.i);
(double[][] Embedding,double Explained) Embed(string[] docs,int dim) {
    var bow=docs.Select(t=>{var a=new double[VOCAB.Length];foreach(var word in t.Split(' ',StringSplitOptions.RemoveEmptyEntries))if(vocabIndex.TryGetValue(word,out int j))a[j]++;return a;}).ToArray();
    var df=Enumerable.Range(0,VOCAB.Length).Select(j=>bow.Count(a=>a[j]>0)).ToArray();
    var weighted=bow.Select(a=>Unit(a.Select((v,j)=>v>0?(1+Math.Log(v))*(1+Math.Log((1.0+docs.Length)/(1+df[j]))):0).ToArray())).ToArray();
    var sv=Svd(weighted,dim);return(sv.Values.Select(Unit).ToArray(),sv.Explained);
}
const int EMB_DIM=32;
var embedded=Embed(metadata,EMB_DIM);double[][] E=embedded.Embedding;
var intra=new List<double>();var inter=new List<double>();
for(int i=0;i<300;i++){int[] same=Enumerable.Range(0,N_ITEMS).Where(j=>j!=i&&itemPrimary[j]==itemPrimary[i]).ToArray();int[] other=Enumerable.Range(0,N_ITEMS).Where(j=>itemPrimary[j]!=itemPrimary[i]).ToArray();intra.Add(Dot(E[i],E[same[rng.Next(same.Length)]]));inter.Add(Dot(E[i],E[other[rng.Next(other.Length)]]));}
Console.WriteLine($"plongement {E.Length}×{E[0].Length} ; variance expliquée {embedded.Explained:F3}");
Console.WriteLine($"similarité cosinus moyenne : même style {intra.Average():F3} ; autres styles {inter.Average():F3}");


plongement 2000×32 ; variance expliquée 0.437


similarité cosinus moyenne : même style 0.482 ; autres styles 0.104


Le contrôle ci-dessus n'est pas décoratif : il établit que l'information de style survit à la réduction de dimension. Sans lui, un mauvais résultat des modèles de contenu serait imputable au plongement, et la comparaison avec les identifiants sémantiques n'aurait aucun sens — les deux modèles reposeraient sur une représentation déjà dégradée.

## Les identifiants sémantiques

La quantification résiduelle transforme un vecteur continu en une **suite de codes** : on découpe l'espace en `K` cellules, on code chaque item par sa cellule, puis on quantifie *le résidu* — l'écart au centre de sa cellule — par un second dictionnaire, et ainsi de suite. Trois niveaux de `K` codes donnent un identifiant de `3 log2 K` bits. Un item devient donc un tuple, et deux items qui partagent les premiers codes sont « voisins de haut niveau ».

C'est le mécanisme du rapport Sona, réduit à sa mécanique : un plongement, des dictionnaires résiduels, des tuples. Rien de propriétaire dans l'opération.

In [5]:
(int[][] Codes,double[][][] Books) ResidualQuantize(double[][] embedding,int levels=3,int bookSize=32,int seed=42) {
    double[][] residual=embedding.Select(a=>(double[])a.Clone()).ToArray();var codes=embedding.Select(_=>new int[levels]).ToArray();var books=new double[levels][][];
    for(int level=0;level<levels;level++){
        using(Py.GIL()){dynamic cluster=Py.Import("sklearn.cluster");using PyObject x=ArrayPy(residual);using PyObject km=cluster.KMeans(n_clusters:bookSize,n_init:4,random_state:seed+level);using PyObject labels=((dynamic)km).fit_predict(x);int[] lab=((dynamic)labels).tolist().As<int[]>();books[level]=((dynamic)km).cluster_centers_.tolist().As<double[][]>();for(int i=0;i<residual.Length;i++){codes[i][level]=lab[i];for(int j=0;j<residual[i].Length;j++)residual[i][j]-=books[level][lab[i]][j];}}
    }return(codes,books);
}
const int N_LEVELS=3,CODEBOOK=32;
var quantized=ResidualQuantize(E,N_LEVELS,CODEBOOK);int[][] sidCodes=quantized.Codes;double[][][] sidBooks=quantized.Books;
double ReconstructionError(double[][] emb,int[][] codes,double[][][] books) => Enumerable.Range(0,emb.Length).Average(i=>Math.Sqrt(Enumerable.Range(0,emb[i].Length).Sum(j=>Math.Pow(emb[i][j]-Enumerable.Range(0,books.Length).Sum(l=>books[l][codes[i][l]][j]),2))));
Console.WriteLine($"identifiants {N_LEVELS}×{CODEBOOK} = {N_LEVELS*Math.Log2(CODEBOOK):F1} bits/item idéaux ; vecteur float32 {EMB_DIM*32} bits (stockage double utilisé ici : {EMB_DIM*64} bits)");
foreach(int i in Pick(Enumerable.Range(0,N_ITEMS).ToArray(),4))Console.WriteLine($"item {i,4} | id ({string.Join(',',sidCodes[i])}) | style {styleNames[itemPrimary[i]]}");
Console.WriteLine($"erreur résiduelle moyenne {ReconstructionError(E,sidCodes,sidBooks):F4} ; premier niveau couvre {sidCodes.Select(a=>a[0]).Distinct().Count()}/{CODEBOOK} codes");
Console.WriteLine($"Dictionnaires : {N_LEVELS*CODEBOOK*EMB_DIM} scalaires, coût non inclus dans les bits/item idéaux.");


identifiants 3×32 = 15.0 bits/item idéaux ; vecteur float32 1024 bits (stockage double utilisé ici : 2048 bits)


item   48 | id (4,17,9) | style ambient


item 1170 | id (31,28,11) | style rock


item 1005 | id (23,21,16) | style reggae


item  678 | id (20,0,26) | style electro


erreur résiduelle moyenne 0.3558 ; premier niveau couvre 32/32 codes


Dictionnaires : 3072 scalaires, coût non inclus dans les bits/item idéaux.



(9,171): info CS9236: Compiling requires binding the lambda expression at least 100 times. Consider declaring the lambda expression with explicit parameter types, or if the containing method call is generic, consider using explicit type arguments.

(9,230): info CS9236: Compiling requires binding the lambda expression at least 1300 times. Consider declaring the lambda expression with explicit parameter types, or if the containing method call is generic, consider using explicit type arguments.



Deux propriétés de la sortie comptent pour la suite. D'abord, la quantification **est** une compression avec perte : l'erreur résiduelle moyenne n'est pas nulle, et c'est son prix qu'il faudra mettre en regard de ce qu'elle rapporte. Ensuite, le premier niveau joue le rôle d'un **regroupement grossier** : deux items qui partagent leur premier code sont traités comme proches parentes par tout modèle qui range dans cet espace, y compris s'ils ne se ressemblent pas finement.

## Les quatre lectures d'un même catalogue

Les modèles comparés ne sont pas quatre variantes d'une même idée, mais quatre réponses à la question « d'où vient le signal ? » :

| Modèle | Signal utilisé | Comportement attendu sur un item froid |
|---|---|---|
| **Populaire** | comptage global des interactions | score zéro avant départage pour un item jamais vu |
| **Collaboratif** (SVD tronquée sur la matrice binaire) | co-occurrences d'interactions | aucun signal d’interaction — la colonne d’entraînement est vide |
| **Contenu** (cosinus au profil) | étiquettes de l'item contre étiquettes consommées | pleine capacité — le contenu existe sans interaction |
| **Identifiants sémantiques** | codes quantifiés contre distribution de codes consommés | pleine capacité, mais à travers la compression |

La baseline de contenu est délibérément **forte** : c'est elle qui rend le résultat des identifiants sémantiques interprétable. Si les identifiants ne battaient qu'un modèle collaboratif aveugle aux items froids, on n'aurait rien appris sur la quantification — seulement sur le démarrage à froid.

In [6]:
double[][] ContentScores(double[][] embedding,HashSet<int>[] histories) {
    var result=histories.Select(_=>new double[embedding.Length]).ToArray();
    for(int u=0;u<histories.Length;u++){var profile=new double[embedding[0].Length];foreach(int i in histories[u])for(int d=0;d<profile.Length;d++)profile[d]+=embedding[i][d]/Math.Max(1,histories[u].Count);profile=Unit(profile);for(int i=0;i<embedding.Length;i++)result[u][i]=Dot(profile,embedding[i]);}return result;
}
double[][] SidScores(int[][] codes,HashSet<int>[] histories,int bookSize,double alpha=.5) {
    int levels=codes[0].Length,n=codes.Length;var prior=new double[levels][];
    for(int l=0;l<levels;l++){var c=new double[bookSize];foreach(var row in codes)c[row[l]]++;prior[l]=c.Select(v=>(v+1)/(n+(double)bookSize)).ToArray();}
    var result=histories.Select(_=>new double[n]).ToArray();
    for(int u=0;u<histories.Length;u++){if(histories[u].Count==0)continue;for(int l=0;l<levels;l++){var c=new double[bookSize];foreach(int i in histories[u])c[codes[i][l]]++;for(int i=0;i<n;i++){int q=codes[i][l];double p=(c[q]+alpha*prior[l][q]*bookSize)/(histories[u].Count+alpha*bookSize);result[u][i]+=Math.Log(p)-Math.Log(prior[l][q]);}}}return result;
}
var X=seen.Select(s=>Enumerable.Range(0,N_ITEMS).Select(i=>s.Contains(i)?1d:0d).ToArray()).ToArray();
var counts=Enumerable.Range(0,N_ITEMS).Select(i=>seen.Count(s=>s.Contains(i))).ToArray();
var S_pop=seen.Select(_=>counts.Select(v=>(double)v).ToArray()).ToArray();
var collaborative=Svd(X,32,true);var S_col=new double[N_USERS][];
for(int u=0;u<N_USERS;u++){S_col[u]=new double[N_ITEMS];for(int i=0;i<N_ITEMS;i++)for(int j=0;j<32;j++)S_col[u][i]+=collaborative.Values[u][j]*collaborative.Components[j][i];}
var S_content=ContentScores(E,seen);var S_sid=SidScores(sidCodes,seen,CODEBOOK);
var MODELS=new Dictionary<string,double[][]>{{"Populaire",S_pop},{"Collaboratif (SVD)",S_col},{"Contenu (cosinus)",S_content},{"Identifiants sémantiques",S_sid}};
Console.WriteLine("quatre matrices de score : "+string.Join(" ; ",MODELS.Select(p=>$"{p.Key}={p.Value.Length}×{p.Value[0].Length}")));
Console.WriteLine($"items froids : {coldItems.Count(i=>counts[i]==0)}/{nItemsCold} colonnes train vides ; max score collaboratif froid {coldItems.Max(i=>S_col.Max(a=>Math.Abs(a[i]))):G4}");


quatre matrices de score : Populaire=800×2000 ; Collaboratif (SVD)=800×2000 ; Contenu (cosinus)=800×2000 ; Identifiants sémantiques=800×2000


items froids : 300/300 colonnes train vides ; max score collaboratif froid 3.227E-07


La dernière ligne de la sortie est le mécanisme du problème, vérifié plutôt que raconté : les colonnes des items froids sont vides, donc le modèle collaboratif ne dispose d’aucun signal collaboratif pour les distinguer : le modèle idéal donne zéro, et la SVD numérique peut laisser de petits résidus. La cellule suivante mesure les conséquences, et elle mesure aussi ce qu'un classement au hasard donnerait, pour que les chiffres soient lus contre une référence et non dans le vide.

In [7]:
const int K=20;
void ValidateRanks(double[][] scores,IReadOnlyDictionary<int,int> targets,HashSet<int>[] histories,int k) {
    if(k<1||scores.Length==0||targets.Count==0||histories.Length!=scores.Length)throw new ArgumentException("k positif, cibles et matrices non vides requis");int n=scores[0].Length;
    if(n==0||scores.Any(a=>a.Length!=n||a.Any(v=>!double.IsFinite(v)))||histories.Any(s=>s.Any(i=>i<0||i>=n)))throw new ArgumentException("scores finis rectangulaires et indices valides requis");
    foreach(var pair in targets)if(pair.Key<0||pair.Key>=scores.Length||pair.Value<0||pair.Value>=n||histories[pair.Key].Contains(pair.Value))throw new ArgumentException("cible absente, hors bornes ou déjà vue");
}
int[] TopK(double[] scores,HashSet<int> history,int k) => Enumerable.Range(0,scores.Length).Where(i=>!history.Contains(i)).OrderByDescending(i=>scores[i]).ThenBy(i=>i).Take(k).ToArray();
bool[] HitVector(double[][] scores,IReadOnlyDictionary<int,int> targets,HashSet<int>[] histories,int k=K){ValidateRanks(scores,targets,histories,k);return targets.Select(p=>TopK(scores[p.Key],histories[p.Key],k).Contains(p.Value)).ToArray();}
(double Recall,double Ndcg) RankMetrics(double[][] scores,IReadOnlyDictionary<int,int> targets,HashSet<int>[] histories,int k=K){ValidateRanks(scores,targets,histories,k);double hits=0,ndcg=0;foreach(var pair in targets){int rank=Array.IndexOf(TopK(scores[pair.Key],histories[pair.Key],k),pair.Value);if(rank>=0){hits++;ndcg+=1/Math.Log2(rank+2);}}return(hits/targets.Count,ndcg/targets.Count);}
double RandomRecall(IReadOnlyDictionary<int,int> targets,HashSet<int>[] histories,int n,int k)=>targets.Keys.Average(u=>Math.Min(k,n-histories[u].Count)/(double)(n-histories[u].Count));
Console.WriteLine($"hasard analytique Recall@{K} : chaud {RandomRecall(warmTarget,seen,N_ITEMS,K):F4} ; froid {RandomRecall(coldTarget,seen,N_ITEMS,K):F4} (moyennes des k/n individuels)");
Console.WriteLine($"{"modèle",-28} {"R chaud %",10} {"NDCG chaud %",13} {"R froid %",10} {"NDCG froid %",13}");
foreach(var model in MODELS){var w=RankMetrics(model.Value,warmTarget,seen);var c=RankMetrics(model.Value,coldTarget,seen);Console.WriteLine($"{model.Key,-28} {100*w.Recall,10:F2} {100*w.Ndcg,13:F2} {100*c.Recall,10:F2} {100*c.Ndcg,13:F2}");}
var tiedScores=new[]{new[]{0d,7,4,0,0,9}};var tiedSeen=new[]{new HashSet<int>{5}};var tiedTarget=new Dictionary<int,int>{{0,0}};
if(RankMetrics(tiedScores,tiedTarget,tiedSeen,3).Recall!=1||TopK(tiedScores[0],tiedSeen[0],99).Contains(5))throw new Exception("contrat des candidats");
bool seenRejected=false;try{RankMetrics(tiedScores,new Dictionary<int,int>{{0,5}},tiedSeen,99);}catch(ArgumentException){seenRejected=true;}if(!seenRejected)throw new Exception("cible vue acceptée");
Console.WriteLine("Contre-exemple contrôlé : un item froid de score constant peut entrer au top-k par ex aequo ; une cible vue est refusée, k est plafonné aux candidats.");


hasard analytique Recall@20 : chaud 0.0102 ; froid 0.0102 (moyennes des k/n individuels)


modèle                        R chaud %  NDCG chaud %  R froid %  NDCG froid %


Populaire                          4.38          1.54       0.00          0.00


Collaboratif (SVD)                 3.50          1.30       0.00          0.00


Contenu (cosinus)                  1.00          0.37       1.75          0.70


Identifiants sémantiques           1.38          0.36       1.75          0.58


Contre-exemple contrôlé : un item froid de score constant peut entrer au top-k par ex aequo ; une cible vue est refusée, k est plafonné aux candidats.


Le tableau confronte les deux populations sur les mêmes candidats. La référence du hasard moyenne les rapports individuels ; le contre-exemple distingue manque de signal et impossibilité de hit.


<a id="exercice-1-deux-niveaux"></a>

### Exercice 1 — Deux niveaux suffisent-ils ?

Quantifiez E en deux niveaux de 32 codes, reconstruisez les scores PMI et comparez le Recall@20 froid à celui des trois niveaux. Complétez une fonction `(embedding, historiques, niveaux, taille_livre)` qui renvoie un `QuantStudent` avec `Codes` et `Scores`.

- Indice : les livres résiduels se réapprennent à chaque niveau ; le lissage utilise le prior du catalogue.
- Étape 1 : reconstruire la chaîne de quantification et de scores.
- Étape 2 : vérifier formes, valeurs finies et absence de fuite d’interactions froides.
- Étape 3 : mesurer le rappel sur le catalogue complet ; un écart n’est pas une supériorité démontrée.

Les gardes acceptent les permutations cohérentes des noms de codes, contrôlent leurs partitions et deux tailles de fixtures, puis comparent les scores à leur formule. Le nombre de niveaux et la taille des livres sont des paramètres effectifs.


In [8]:
record QuantStudent(int[][] Codes,double[][] Scores);
// TODO étudiant : construire les codes et scores pour le nombre de niveaux demandé.
Func<double[][],HashSet<int>[],int,int,QuantStudent> resultEx1 = null;
void CheckScores(double[][] actual,double[][] expected) {
    if(actual==null||actual.Length!=expected.Length||actual.Where((a,i)=>a==null||a.Length!=expected[i].Length||a.Where((v,j)=>!double.IsFinite(v)||Math.Abs(v-expected[i][j])>1e-7).Any()).Any())throw new Exception("matrice de scores incorrecte");
}
void ValidateQuant(Func<double[][],HashSet<int>[],int,int,QuantStudent> f) {
    foreach(int levels in new[]{1,2}){
        var emb=E.Take(levels==1?48:64).Select(a=>a.Take(8).ToArray()).ToArray();var histories=new[]{new HashSet<int>{0,2,5},new HashSet<int>{3,4},new HashSet<int>()};
        var expected=ResidualQuantize(emb,levels,4);var answer=f(emb,histories,levels,4);
        if(answer==null||answer.Codes==null||answer.Codes.Length!=emb.Length||answer.Codes.Any(a=>a==null||a.Length!=levels||a.Any(c=>c<0||c>=4)))throw new Exception("codes invalides");
        for(int l=0;l<levels;l++)for(int i=0;i<emb.Length;i++)for(int j=0;j<emb.Length;j++)if((answer.Codes[i][l]==answer.Codes[j][l])!=(expected.Codes[i][l]==expected.Codes[j][l]))throw new Exception("partition KMeans incorrecte");
        CheckScores(answer.Scores,SidScores(answer.Codes,histories,4));
    }
}
if(resultEx1==null)Console.WriteLine("Exercice 1 à compléter : codes à deux niveaux et scores PMI ; les réponses fournies seront contrôlées sur deux fixtures.");
else {ValidateQuant(resultEx1);var answer=resultEx1(E,seen,2,CODEBOOK);Console.WriteLine($"Recall@20 froid deux niveaux {100*RankMetrics(answer.Scores,coldTarget,seen).Recall:F2}% ; trois niveaux {100*RankMetrics(S_sid,coldTarget,seen).Recall:F2}%");}


Exercice 1 à compléter : codes à deux niveaux et scores PMI ; les réponses fournies seront contrôlées sur deux fixtures.


## Les écarts sont-ils plus grands que le bruit ?

Un rappel de 1,88 % contre 1,63 % ne veut rien dire tant qu'on ne sait pas de combien deux modèles **identiques** varieraient sur le même échantillon. Les deux contrastes qui portent la conclusion sont donc testés appariés — mêmes utilisateurs, mêmes cibles, seule change la matrice de score.

Deux instruments, complémentaires :

- le **test exact de McNemar** sur les paires discordantes (là où un modèle atteint la cible et l'autre non). Il ne suppose rien sur la forme de la distribution des rappels, seulement que les utilisateurs sont indépendants ;
- l'**intervalle de confiance bootstrap** de la différence de rappel, qui donne l'ordre de grandeur en points de pourcentage — la grandeur que le lecteur compare intuitivement.


In [9]:
(double Lo,double Hi) BootstrapDiff(bool[] a,bool[] b,int replicates=4000,int seed=7) {
    if(a.Length==0||a.Length!=b.Length||replicates<1)throw new ArgumentException("vecteurs appariés non vides requis");var random=new Random(seed);var differences=new double[replicates];
    for(int r=0;r<replicates;r++){double sum=0;for(int j=0;j<a.Length;j++){int i=random.Next(a.Length);sum+=(a[i]?1:0)-(b[i]?1:0);}differences[r]=sum/a.Length;}Array.Sort(differences);
    double Quantile(double p){double q=p*(differences.Length-1);int lo=(int)Math.Floor(q),hi=(int)Math.Ceiling(q);return differences[lo]+(q-lo)*(differences[hi]-differences[lo]);}return(Quantile(.025),Quantile(.975));
}
double McNemar(int aOnly,int bOnly){if(aOnly<0||bOnly<0)throw new ArgumentException("comptages négatifs");int n=aOnly+bOnly;return n==0?1:Math.Min(1,2*Binomial.CDF(.5,n,Math.Min(aOnly,bOnly)));}
var contrasts=new[]{("Identifiants vs Contenu (froid)","Identifiants sémantiques","Contenu (cosinus)",coldTarget),("Identifiants vs Contenu (chaud)","Identifiants sémantiques","Contenu (cosinus)",warmTarget),("Populaire vs Contenu (chaud)","Populaire","Contenu (cosinus)",warmTarget),("Collaboratif vs Populaire (chaud)","Collaboratif (SVD)","Populaire",warmTarget)};
Console.WriteLine($"{"contraste",-37} {"A %",7} {"B %",7} {"écart pts",10} {"IC95% pts",20} {"McNemar p",12}");
foreach(var (label,aName,bName,targets) in contrasts){var a=HitVector(MODELS[aName],targets,seen);var b=HitVector(MODELS[bName],targets,seen);var ci=BootstrapDiff(a,b);int ao=a.Zip(b).Count(p=>p.First&&!p.Second),bo=a.Zip(b).Count(p=>!p.First&&p.Second);double p=McNemar(ao,bo),witness=1;
    if(ao+bo>0)using(Py.GIL()){dynamic ss=Py.Import("scipy.stats");witness=(double)ss.binomtest(Math.Max(ao,bo),ao+bo,.5).pvalue;}
    if(Math.Abs(p-witness)>1e-12)throw new Exception("désaccord MathNet/SciPy");double ma=a.Count(v=>v)/(double)a.Length,mb=b.Count(v=>v)/(double)b.Length;
    Console.WriteLine($"{label,-37} {100*ma,7:F2} {100*mb,7:F2} {100*(ma-mb),+10:F2} [{100*ci.Lo,+7:F2};{100*ci.Hi,+7:F2}] {p,12:G5} (discordances {ao}/{bo}, n={a.Length}, ΔSciPy={Math.Abs(p-witness):G3})");}
if(BootstrapDiff(new[]{true,false,true},new[]{true,false,true})!=(0d,0d)||McNemar(7,3)!=McNemar(3,7)||McNemar(0,0)!=1||!(McNemar(0,800)>0))throw new Exception("contrats statistiques");
Console.WriteLine("4000 réplications appariées par contraste ; IC et p-values individuels, sans correction de multiplicité. Témoins identique/symétrie/800 discordances : OK.");


contraste                                 A %     B %  écart pts            IC95% pts    McNemar p


Identifiants vs Contenu (froid)          1.75    1.75       0.00 [  -1.13;   1.25]            1 (discordances 12/12, n=799, ΔSciPy=0)


Identifiants vs Contenu (chaud)          1.38    1.00       0.38 [  -0.50;   1.25]      0.58105 (discordances 8/5, n=800, ΔSciPy=1.44E-15)


Populaire vs Contenu (chaud)             4.38    1.00       3.37 [   1.88;   5.00]   4.1934E-05 (discordances 35/8, n=800, ΔSciPy=1.36E-19)


Collaboratif vs Populaire (chaud)        3.50    4.38      -0.87 [  -2.62;   0.75]      0.38169 (discordances 20/27, n=800, ΔSciPy=5.66E-15)


4000 réplications appariées par contraste ; IC et p-values individuels, sans correction de multiplicité. Témoins identique/symétrie/800 discordances : OK.


## Ce que les tests autorisent à écrire

Sur ces quatre contrastes exploratoires, seul le populaire contre le contenu chez les items chauds franchit ici le seuil individuel de 5 % : +3,37 points de rappel, IC bootstrap [1,88 ; 5,00], McNemar p = 4,1934×10⁻⁵. Les 35 succès propres au populaire contre 8 au contenu expliquent ce verdict. Les autres comparaisons restent non concluantes :

- identifiants contre contenu froid : mêmes rappels moyens de 1,75 %, différence 0,00 point, IC [−1,13 ; +1,25], p = 1 ; leurs succès ne sont pas les mêmes, avec 12 discordances dans chaque sens ;
- identifiants contre contenu chaud : +0,38 point, IC [−0,50 ; +1,25], p = 0,58105 ;
- collaboratif contre populaire chaud : −0,87 point, IC [−2,62 ; +0,75], p = 0,38169.

Ces valeurs viennent du corpus .NET courant : ses tirages diffèrent de NumPy malgré les mêmes paramètres. Les p-values sont celles du test binomial exact à p=1/2 sur les discordances ; MathNet et le vrai SciPy concordent à moins de 10⁻¹². Le bootstrap rééchantillonne les mêmes utilisateurs pour A et B, avec 4000 réplications par contraste ; les IC et p-values restent individuels, sans garantie simultanée sur les quatre tests.

Un écart de rappel affiché seul ne porte pas sa propre incertitude. Le nombre de succès discordants importe : c’est pourquoi un petit écart n’a pas un seuil universel de « deux points et demi ». Un intervalle contenant zéro laisse plusieurs signes compatibles avec ces données ; ce n’est ni une preuve d’équivalence ni une supériorité cachée.

Les colonnes d’entraînement froides sont exactement vides, mais cette propriété ne garantit pas un rappel nul pour tout classement. La SVD float32 laisse ici des résidus de score jusqu’à 3,227×10⁻⁷ ; le modèle idéal n’a aucun signal collaboratif sur ces colonnes. Le top-k dépend encore des candidats mieux classés et des ex aequo. Le contre-exemple du classement construit un hit froid malgré un score nul. Les zéros observés ici sont des résultats de ce corpus, pas une impossibilité générale.


Deux lectures se dégagent, et elles ne vont pas dans le même sens — c’est le résultat du carnet.

**Sur les items froids**, populaire et collaboratif obtiennent 0 % dans ce run. Leurs colonnes d’interactions sont vides ; le contenu et les identifiants utilisent au contraire les métadonnées du catalogue et atteignent chacun 1,75 % de rappel, contre environ 1,02 % attendu pour un rang uniforme. Ce dernier écart est descriptif : aucun test contenu-contre-hasard n’a été exécuté ici. L’absence de signal collaboratif explique la difficulté, mais ne constitue pas un théorème de rappel zéro : le départage et l’univers de candidats comptent.

**Sur les items chauds**, le populaire atteint 4,38 %, le collaboratif 3,50 %, les identifiants 1,38 % et le contenu 1,00 %. Le générateur combine affinité de style et popularité log-normale ; cette dernière transporte donc une partie du signal. Une popularité uniforme pourrait modifier ce classement, ce qui demanderait une autre expérience.

Pour la question posée, les identifiants ne montrent aucun gain froid sur le contenu direct : le rappel moyen est le même, avec des succès différents et p=1. Leur avance chaude de 0,38 point n’est pas établie par le test apparié (p≈0,581). La quantification n’est pas un gain gratuit : elle change les représentations et leurs classements, sans supériorité démontrée à cette échelle.


<a id="exercice-2-prefixes"></a>

### Exercice 2 — Le partage du premier code aide-t-il ?

Parmi les cibles froides atteintes par les identifiants, mesurez la part qui partage son premier code avec au moins un item consommé. Complétez une fonction `(scores, cibles, historiques, codes, k)` renvoyant un `PrefixStudent(Hits, Shared, Fraction, RandomReference)`.

- Indice : commencez par le top-k des candidats non vus, avec départage stable.
- Étape 1 : isoler les utilisateurs où la cible est atteinte.
- Étape 2 : compter les premiers codes partagés, sans compter plusieurs fois un même hit.
- Étape 3 : calculer aussi le témoin aléatoire déclaré : pour ce même ensemble de succès, tirer un item uniformément dans le catalogue et demander si son premier code appartient aux codes consommés. La probabilité est la masse des codes consommés dans la distribution empirique du catalogue, puis sa moyenne parmi les succès.

Sans succès, les deux fractions sont non mesurables (`null`), avec comptes nuls. Les gardes comprennent des succès identiques dont seuls les préfixes changent : compter les hits ne suffit pas.


In [10]:
record PrefixStudent(int Hits,int Shared,double? Fraction,double? RandomReference);
// TODO étudiant : mesurer le partage du premier code parmi les cibles atteintes et le témoin aléatoire.
Func<double[][],Dictionary<int,int>,HashSet<int>[],int[][],int,PrefixStudent> resultEx2 = null;
void ValidatePrefix(Func<double[][],Dictionary<int,int>,HashSet<int>[],int[][],int,PrefixStudent> f) {
    var codes=new[]{new[]{0},new[]{1},new[]{0},new[]{2},new[]{1}};
    var histories=new[]{new HashSet<int>{1,4},new HashSet<int>{0,2},new HashSet<int>{1}};
    var targets=new Dictionary<int,int>{{0,0},{1,3},{2,4}};
    var scores=new[]{new[]{10d,0,0,0,0},new[]{0d,0,0,10,0},new[]{0d,0,0,0,10}};
    var a=f(scores,targets,histories,codes,1);
    if(a==null||a.Hits!=3||a.Shared!=1||!a.Fraction.HasValue||!double.IsFinite(a.Fraction.Value)||Math.Abs(a.Fraction.Value-1d/3)>1e-12||!a.RandomReference.HasValue||!double.IsFinite(a.RandomReference.Value)||Math.Abs(a.RandomReference.Value-.4)>1e-12)throw new Exception("partage ou référence aléatoire incorrect");
    var sameCodes=codes.Select(_=>new[]{0}).ToArray();var c=f(scores,targets,histories,sameCodes,1);
    if(c==null||c.Hits!=3||c.Shared!=3||c.Fraction!=1||c.RandomReference!=1)throw new Exception("le résultat doit dépendre des préfixes, pas seulement des succès");
    var miss=new[]{new[]{0d,0,9,0,0},new[]{0d,9,0,0,0},new[]{9d,0,0,0,0}};var b=f(miss,targets,histories,codes,1);
    if(b==null||b.Hits!=0||b.Shared!=0||b.Fraction.HasValue||b.RandomReference.HasValue)throw new Exception("zéro succès doit être non mesurable");
}
if(resultEx2==null)Console.WriteLine("Exercice 2 à compléter : part de préfixes communs ET témoin uniforme sur les items du catalogue ; aucune fraction si aucun succès.");
else {ValidatePrefix(resultEx2);var r=resultEx2(S_sid,coldTarget,seen,sidCodes,K);Console.WriteLine($"succès {r.Hits}, partagés {r.Shared}, fraction {r.Fraction}, témoin aléatoire {r.RandomReference}");}


Exercice 2 à compléter : part de préfixes communs ET témoin uniforme sur les items du catalogue ; aucune fraction si aucun succès.


## Le prix de la compression

Ranger dans un espace d'identifiants n'est pas gratuit : la quantification détruit de l'information, et le nombre de codes par niveau règle directement cette perte. La cellule suivante fait varier le dictionnaire à nombre de niveaux fixé, et reporte le coût en bits par item en regard du rappel obtenu sur les items froids. Le but n'est pas de trouver « le bon » réglage, mais de voir la **forme de la courbe** : si le rappel est plat sur une large plage de tailles, la compression est bon marché ; s'il s'effondre, l'identifiant sémantique n'est utilisable qu'à un coût qu'il faut alors nommer.

In [11]:
var sweep=new List<(int Book,double Bits,double Cold,double Warm,double Ndcg)>();
foreach(int cb in new[]{8,16,32,64}){var q=ResidualQuantize(E,N_LEVELS,cb);var scores=SidScores(q.Codes,seen,cb);var cold=RankMetrics(scores,coldTarget,seen);var warm=RankMetrics(scores,warmTarget,seen);sweep.Add((cb,N_LEVELS*Math.Log2(cb),cold.Recall,warm.Recall,cold.Ndcg));}
Console.WriteLine($"{"codes/niveau",14} {"bits/item",11} {"Recall froid %",16} {"Recall chaud %",16} {"NDCG froid %",15}");
foreach(var r in sweep)Console.WriteLine($"{r.Book,14} {r.Bits,11:F1} {100*r.Cold,16:F2} {100*r.Warm,16:F2} {100*r.Ndcg,15:F2}");
var best=sweep.OrderByDescending(r=>r.Cold).First();Console.WriteLine($"meilleur rappel froid observé : {best.Book} codes, {best.Bits:F1} bits/item idéaux ; ce balayage descriptif ne sélectionne pas un modèle validé.");


  codes/niveau   bits/item   Recall froid %   Recall chaud %    NDCG froid %


             8         9.0             1.63             1.62            0.62


            16        12.0             2.25             1.25            0.80


            32        15.0             1.75             1.38            0.58


            64        18.0             1.38             1.38            0.52


meilleur rappel froid observé : 16 codes, 12.0 bits/item idéaux ; ce balayage descriptif ne sélectionne pas un modèle validé.


Le rappel froid n’augmente pas régulièrement avec la taille du livre. Il vaut 1,63 % à 8 codes (9 bits idéaux), culmine à 2,25 % à 16 codes (12 bits), puis passe à 1,75 % à 32 et 1,38 % à 64 (18 bits). Ajouter des codes n’achète donc pas ici un gain monotone.

Ce balayage ne valide pas un meilleur réglage : il réutilise les mêmes cibles et ne comporte pas d’incertitude appariée pour chacun de ces écarts. Les fractions de point peuvent refléter le bruit d’échantillonnage et l’estimation des profils. Il montre la bonne question — la qualité contre le coût des codes — sans transformer son maximum observé en hyperparamètre testé sur un nouvel échantillon.

La quantification résiduelle est un levier de stockage et de partage de préfixe, qui perd de l’information. Les 9/12/15/18 bits par item sont des coûts de codage idéaux des indices ; les tableaux int .NET, leurs objets et les livres de centres sont exclus. À 3×32 codes et 32 dimensions, les livres contiennent 3072 scalaires. Le plongement est stocké ici en double (2048 bits par item), tandis que 1024 bits est la référence hypothétique float32. Ces chiffres ne sont pas une mesure complète de mémoire ni de coût de serving.

Un identifiant devient une suite de symboles qu’un autre modèle pourrait produire. Ce carnet ne fait que scorer : il n’entraîne ni encodeur contrastif collaboratif ni décodeur autorégressif du rapport Sona, et ne reproduit pas sa compression hiérarchique d’historique.


<a id="exercice-3-bruit"></a>

### Exercice 3 — Un contenu moins informatif

Le protocole `NoiseTexts` remplace indépendamment chaque mot par un mot uniforme du vocabulaire avec probabilité configurable (1/3 par défaut, graine 7). Recalculez plongement et modèles, puis mesurez les deux rappels froids. Aucun recul ni ordre de robustesse n’est garanti sur un seul tirage.

Complétez une fonction `(textes_bruites, historiques, dimension, niveaux, taille_livre)` renvoyant un `NoiseStudent(Content, Sid)`.

- Indice : réapprenez IDF/SVD et les livres sur le même catalogue bruité ; gardez les interactions et les cibles.
- Étape 1 : reconstruire les représentations à partir des textes reçus, pas du E initial.
- Étape 2 : reconstruire profils de contenu et scores PMI.
- Étape 3 : comparer les rappels au témoin sans bruit et interpréter le signe observé.

Deux petits corpus, sans bruit puis bruités, contrôlent réellement les matrices finies et leurs dimensions ; la mesure finale conserve le budget 2000 items et 800 utilisateurs.


In [12]:
record NoiseStudent(double[][] Content,double[][] Sid);
// TODO étudiant : recalculer plongement puis les deux scores à partir des textes bruités.
Func<string[],HashSet<int>[],int,int,int,NoiseStudent> resultEx3 = null;
string[] NoiseTexts(string[] docs,double fraction=1d/3,int seed=7) {
    if(!double.IsFinite(fraction)||fraction<0||fraction>1)throw new ArgumentException("fraction entre 0 et 1 requise");var r=new Random(seed);
    return docs.Select(t=>string.Join(' ',t.Split(' ',StringSplitOptions.RemoveEmptyEntries).Select(w=>r.NextDouble()<fraction?VOCAB[r.Next(VOCAB.Length)]:w))).ToArray();
}
void ValidateNoise(Func<string[],HashSet<int>[],int,int,int,NoiseStudent> f) {
    foreach(double fraction in new[]{0d,1d/3}){
        var texts=NoiseTexts(metadata.Take(64).ToArray(),fraction,19);var histories=new[]{new HashSet<int>{0,1,2},new HashSet<int>{3,4},new HashSet<int>{5}};var emb=Embed(texts,8).Embedding;var q=ResidualQuantize(emb,2,4);var a=f(texts,histories,8,2,4);
        if(a==null)throw new Exception("réponse de bruit absente");CheckScores(a.Content,ContentScores(emb,histories));CheckScores(a.Sid,SidScores(q.Codes,histories,4));
    }
}
if(resultEx3==null)Console.WriteLine("Exercice 3 à compléter : deux matrices recalculées ; fraction de bruit configurable et pas de baisse de rappel présupposée.");
else {ValidateNoise(resultEx3);var r=resultEx3(NoiseTexts(metadata),seen,EMB_DIM,N_LEVELS,CODEBOOK);Console.WriteLine($"Recall froid contenu bruité {100*RankMetrics(r.Content,coldTarget,seen).Recall:F2}% (initial {100*RankMetrics(S_content,coldTarget,seen).Recall:F2}%) ; SID bruité {100*RankMetrics(r.Sid,coldTarget,seen).Recall:F2}% (initial {100*RankMetrics(S_sid,coldTarget,seen).Recall:F2}%)");}


Exercice 3 à compléter : deux matrices recalculées ; fraction de bruit configurable et pas de baisse de rappel présupposée.


## Ce que ce carnet établit, et ce qu’il ne dit pas

**Établi par la mesure** : le contenu du catalogue reste exploitable sur les 300 items froids malgré leurs colonnes d’interactions vides. Le populaire et le collaboratif ont ici zéro rappel froid, les deux modèles de contenu 1,75 % ; ce zéro dépend du classement et ne découle pas à lui seul de colonnes vides. Les quatre contrastes appariés séparent effets observés et incertitude ; seul populaire-contre-contenu chaud franchit le seuil individuel choisi dans ce run.

**Non établi** : que les identifiants quantifiés améliorent le contenu direct. Même rappel froid ne veut pas dire mêmes succès, et l’avance chaude reste non concluante. L’erreur de quantification est mesurée, les tailles de livres sont balayées au même budget ; leur courbe descriptive n’est pas une sélection indépendante de modèle.

Ce qui serait trompeur : présenter ces résultats comme ceux d’un recommandateur génératif entraîné, transporter les scores absolus sur un catalogue réel, ou prétendre reproduire la mémoire hiérarchique de Sona. Le corpus est synthétique, le contenu est informatif par construction et les profils sont de simples moyennes. La SVD réelle est calculée en float32 pour le collaboratif, puis le produit U/V est accumulé en double C# ; ces arrondis et les RNG diffèrent du Python.

Le changement de représentation reste tangible : trois indices parmi 32 correspondent à 15 bits idéaux par item, hors livres et structures effectives. Ils rendent une sortie symbolique possible pour un modèle génératif futur ; ce n’est pas en soi un gain de rappel démontré ici.


## Exercices

Les trois exercices ouverts sont distribués après les notions qu’ils utilisent : [deux niveaux](#exercice-1-deux-niveaux), [partage du premier code](#exercice-2-prefixes), puis [contenu bruité](#exercice-3-bruit). Les stubs s’exécutent sans erreur ; une réponse renseignée déclenche de vrais contrôles sur plusieurs fixtures et les données du carnet.


## À retenir

Le contenu rend les items froids visibles ; la quantification comprime une représentation avec perte, sans garantir un gain de rappel. Les scores froids idéaux nuls ne constituent pas une impossibilité générale de hit : candidats, k, arrondis et départage comptent. Les quatre tests sont appariés et exploratoires ; les p-values et IC sont individuels, non corrigés pour la multiplicité. Les 15 bits sont une limite de codage des trois indices, hors livres et stockage effectif.

## Vérifiez votre compréhension

1. Pourquoi un score froid nul peut-il néanmoins donner un hit ?
<details><summary>Réponse</summary>Le top-k dépend des concurrents et des ex aequo. S’il reste peu de candidats mieux classés, un item de score nul peut entrer ; le contre-exemple le vérifie.</details>

2. Les 15 bits mesurent-ils la mémoire de la structure en RAM ?
<details><summary>Réponse</summary>Non : ils codent idéalement trois indices parmi 32. Les tableaux d’entiers, objets, dictionnaires de centres et profils ne sont pas inclus.</details>

3. Une baisse après bruit est-elle garantie ?
<details><summary>Réponse</summary>Non. Un tirage fini peut redistribuer les rangs dans les deux sens ; il faut mesurer plusieurs réplications pour une conclusion robuste.</details>

## Pour aller plus loin

- [ML-07 — recommandation](ML-07-Recommendation-Python.ipynb) : signal collaboratif et items connus.
- [ML-08 — clustering](ML-08-Clustering-Python.ipynb) : représentations et regroupements.
- [Index ML.NET](README.md) : le parcours et les variantes.


## Références

- **Sona** — équipe Yandex Music, *Sona Technical Report*, [arXiv:2608.11015](https://arxiv.org/abs/2608.11015) : recommandeur génératif mono-modèle, tokenisation sémantique par quantification résiduelle, compression d'historique. Cité pour le paradigme et le vocabulaire ; aucune valeur numérique n'en est reprise.
- [ML-07-Recommendation-Python](ML-07-Recommendation-Python.ipynb) — la factorisation matricielle dont ce carnet mesure la limite froide.
- [ML-08-Clustering-Python](ML-08-Clustering-Python.ipynb) — le précédent du corpus contrôlé à vérité terrain cachée.